In [ ]:
import os
import uuid
from datetime import datetime
import groq
import pdfplumber
from docx import Document
from settings import settings
from langchain_groq import ChatGroq
from langchain_qdrant import QdrantVectorStore
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from pydantic import BaseModel
from typing import List, Dict
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

In [ ]:
from resume_processor import (
    load_resume,
    semantic_chunk_text,
    store_resume_chunks_in_qdrant,
    validate_resume_with_llm,
    evaluate_resume_for_roles
)

# Ensure resume_processor.py is accessible
# Adjust path if needed
# sys.path.append("resume_processor.py")

# Verify file existence
resume_path = r"Resume/aishu-resume.pdf"
print(f"Checking if file exists: {os.path.exists(resume_path)}")

In [ ]:
try:
    resume_text = load_resume(resume_path)
    print("Resume text extracted successfully:")
    print(resume_text[:500])  # Print first 500 characters
except ValueError as ve:
    print(f"Error loading resume: {ve}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
try:
    if 'resume_text' in locals():
        chunks = semantic_chunk_text(resume_text)
        print(f"Number of chunks: {len(chunks)}")
        print("First chunk:")
        print(chunks[0])
    else:
        print("Resume text not available. Run load_resume first.")
except ValueError as ve:
    print(f"Error chunking text: {ve}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
import uuid
from datetime import datetime
from settings import settings
from qdrant_client import QdrantClient
try:
    print(f"Testing Qdrant connection to {settings.QDRANT_URL}")
    client = QdrantClient(url=settings.QDRANT_URL, api_key=settings.QDRANT_API_KEY)
    collections = client.get_collections()
    print("Qdrant connection successful. Collections:", [col.name for col in collections.collections])
except Exception as e:
    print(f"Qdrant connection error: {e}")

In [ ]:


try:
    if 'chunks' in locals():
        metadata = {
            "file_name": os.path.basename(resume_path),
            "uploaded_at": datetime.now().isoformat(),
            "resume_id": str(uuid.uuid4())
        }
        vector_store = store_resume_chunks_in_qdrant(chunks, metadata)
        print("Resume chunks stored in Qdrant successfully.")
    else:
        print("Chunks not available. Run semantic_chunk_text first.")
except ValueError as ve:
    print(f"Error storing chunks: {ve}")


#updated

In [1]:
# test.ipynb - Cell 1: Import functions and libraries
from resume_processor import (
    load_resume,
    sanitize_input,
    semantic_chunk_text,
    store_resume_chunks_in_qdrant,
    validate_resume_with_llm,
    evaluate_resume_for_roles,
    embeddings
)

from settings import settings
import os
from pprint import pprint


c:\vinod\projects\resume-chatbot\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\vinod\projects\resume-chatbot\resume_processor.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model)


In [2]:
# Verify file existence
resume_path = r"Resume/aishu-resume.pdf"
print(f"Checking if file exists: {os.path.exists(resume_path)}")
try:
    resume_text = load_resume(resume_path)
    print("✅ Resume loaded successfully.")
    print(resume_text[:500])  # Show preview
except Exception as e:
    print("❌ Failed to load resume:", e)


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


Checking if file exists: True


CropBox missing from /Page, defaulting to MediaBox


✅ Resume loaded successfully.
POLURU AISHWARYA
Contact: +91- 7989805025
Email: aishwaryavinod.n@gmail.com
OBJECTIVE
To contribute my education and technical skills in management field that enables me to learn new
technologies and work for a progressive organization, which place me to have a professional growth in my
carrier.
EDUCATIONAL QUALIFICATONS:
Course University/board College/school Year of passing Percentage/CGPA
B.Pharm JNTUA Sri Padhmavathi 2024 65%
school of pharmacy
Intermediate Board of intermediate Sri Chaithan


In [3]:
# Cell 3: Sanitize input (optional)
try:
    clean_text = sanitize_input(resume_text)
    print("✅ Sanitized resume:")
    print(clean_text[:500])
except Exception as e:
    print("❌ Failed to sanitize resume:", e)


✅ Sanitized resume:
POLURU AISHWARYA
Contact: +91- 7989805025
Email: aishwaryavinod.n@gmail.com
OBJECTIVE
To contribute my education and technical skills in management field that enables me to learn new
technologies and work for a progressive organization, which place me to have a professional growth in my
carrier.
EDUCATIONAL QUALIFICATONS:
Course University/board College/school Year of passing Percentage/CGPA
B.Pharm JNTUA Sri Padhmavathi 2024 65%
school of pharmacy
Intermediate Board of intermediate Sri Chaithan


In [4]:
# Cell 4: Chunk the resume
try:
    chunks = semantic_chunk_text(clean_text)
    print(f"✅ Chunking successful: {len(chunks)} chunks")
    pprint(chunks[:3])
except Exception as e:
    print("❌ Chunking failed:", e)


✅ Chunking successful: 4 chunks
['POLURU AISHWARYA\n'
 'Contact: +91- 7989805025\n'
 'Email: aishwaryavinod.n@gmail.com\n'
 'OBJECTIVE\n'
 'To contribute my education and technical skills in management field that '
 'enables me to learn new\n'
 'technologies and work for a progressive organization, which place me to have '
 'a professional growth in my\n'
 'carrier.\n'
 'EDUCATIONAL QUALIFICATONS:\n'
 'Course University/board College/school Year of passing Percentage/CGPA\n'
 'B.Pharm JNTUA Sri Padhmavathi 2024 65%\n'
 'school of pharmacy',
 'school of pharmacy\n'
 'Intermediate Board of intermediate Sri Chaithanya Junior 2020 7.47\n'
 'education AP College\n'
 'SSC Board of secondary Geetanjali High 2018 8.7\n'
 'education AP School\n'
 'PROJECT\n'
 'Title: Analytical method development and validation\n'
 'Under the guidelines of Dr. Keerthishika, Dept of pharmaceutical analysis.\n'
 'TECHNICAL SKILLS\n'
 'Good knowledge on MS (word, MS Office, Excel, power point, paper writing)\n'
 '

In [5]:
# Cell X: Store chunks using LangChain
try:
    chunks = semantic_chunk_text(resume_text)
    metadata = {"source": os.path.basename(resume_path), "uploaded_by": "test_user"}
    vector_store = store_resume_chunks_in_qdrant(chunks, metadata)
    print("✅ Resume chunks stored successfully with LangChain QdrantVectorStore.")
except Exception as e:
    print("❌ Error storing resume chunks:", e)


✅ Resume chunks stored successfully with LangChain QdrantVectorStore.


In [6]:
# Cell 6: Validate resume using LLM
try:
    feedback = validate_resume_with_llm(resume_text)
    print("✅ LLM feedback received:")
    print(feedback)
except Exception as e:
    print("❌ LLM validation failed:", e)


✅ LLM feedback received:
- ✅ Contact Info: The resume contains the candidate's phone number and email address.
- ✅ Educational Qualifications: The resume provides a clear outline of the candidate's educational background, including the course, university, college, and year of passing.
- ✅ Project: The inclusion of a project title and brief description is a good start.
- ✅ Technical Skills and Soft Skills: The resume mentions the candidate's technical and soft skills, which is essential for any job application.
- ✅ Industrial Visit: The candidate has included their industrial training experience, which is a valuable addition.
- ✅ Personal Profile: The resume contains the candidate's personal details, such as full name, date of birth, and address.

- ⚠️ Objective: The objective statement is quite generic and does not specifically highlight the candidate's strengths or career goals. It would be better to tailor this section to the job being applied for.
- ⚠️ Project: The project descripti

In [7]:
# Cell 7: Evaluate against mock job roles
job_roles = [
    {
        "role_name": "systems suppot pharmacist",
        
        "description": "Looking for a systems support pharmacist with experience in pharmacy systems and data analysis.",
        "skills": ["Pharma analysis", "chemical Analysis","Excel","ms office"],
        "experience": 0-2,
        "education": "Bachelor's Degree of pharmacy"
    },

]

try:
    results = evaluate_resume_for_roles(resume_text, job_roles, vector_store)
    print("✅ Role evaluation completed.")
    pprint(results)
except Exception as e:
    print("❌ Role evaluation failed:", e)


✅ Role evaluation completed.
[{'evaluation': 'To evaluate the eligibility of the provided resume for the '
                'Systems Support Pharmacist role, I will assess the resume '
                'based on the required skills, experience, and education.\n'
                '\n'
                '**Match Score: 60**\n'
                '\n'
                '**Skills Match: 30/50**\n'
                'The resume mentions "Good knowledge on MS (word, MS Office, '
                'Excel, power point, paper writing)", which covers the MS '
                'Office and Excel required skills. However, it does not '
                'explicitly mention "Pharma analysis" or "chemical Analysis". '
                'The project title "Analytical method development and '
                'validation" under the guidance of a pharmaceutical analysis '
                'department suggests some exposure to pharma analysis, but it '
                'is not a direct match. Therefore, the skills match score